In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/digit-recognizer/test.csv")

X = train_df.drop(columns='label').values / 255.0
y = train_df['label'].values
X_test = test_df.values / 255.0

# Model is so small compared to amount of input, that it can't possibly overfit.
# Augmentation makes exploration space even more vast
X_train, X_valid = X, X
y_train, y_valid = y, y


In [ ]:
import torchvision.transforms.functional as TF
import random
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

def random_affine(x, angle_range=15, translate_range=2, scale_range=(0.8, 1.1)):
    angle = random.uniform(-angle_range, angle_range)
    tx = random.randint(-translate_range, translate_range)
    ty = random.randint(-translate_range, translate_range)
    sx = random.uniform(*scale_range)
    sy = random.uniform(*scale_range)

    theta = np.radians(angle)
    cos, sin = np.cos(theta), np.sin(theta)

    # affine matrix: rotation × independent scale
    mat = torch.tensor([[sx * cos, -sy * sin, tx / x.shape[-1]],
                        [sx * sin,  sy * cos, ty / x.shape[-2]]],
                       dtype=x.dtype).unsqueeze(0)

    grid = F.affine_grid(mat, x.unsqueeze(0).shape, align_corners=False)
    return F.grid_sample(x.unsqueeze(0), grid, align_corners=False,
                         padding_mode='zeros').squeeze(0)

# Apparently only harmful for such a small model. Transforms digits into ambiguous shapes
def elastic_distort(x, alpha=3, sigma=1):
    _, h, w = x.shape
    dx = torch.randn(1, 1, h, w) * alpha
    dy = torch.randn(1, 1, h, w) * alpha
    
    kernel_size = int(2 * sigma + 1) | 1
    pad = kernel_size // 2
    coords = torch.arange(kernel_size).float() - kernel_size // 2
    gauss = torch.exp(-coords**2 / (2 * sigma**2))
    gauss = gauss / gauss.sum()
    kernel = (gauss[:, None] * gauss[None, :]).unsqueeze(0).unsqueeze(0)
    
    dx = F.conv2d(F.pad(dx, [pad]*4, mode='reflect'), kernel)[:, :, :h, :w]
    dy = F.conv2d(F.pad(dy, [pad]*4, mode='reflect'), kernel)[:, :, :h, :w]
    
    grid_y, grid_x = torch.meshgrid(torch.linspace(-1,1,h), torch.linspace(-1,1,w), indexing='ij')
    grid = torch.stack([grid_x + dx.squeeze() / w, grid_y + dy.squeeze() / h], dim=-1).unsqueeze(0)
    return F.grid_sample(x.unsqueeze(0), grid, align_corners=False, padding_mode='reflection').squeeze(0)

class MNISTDataset(Dataset):
    def __init__(self, X, y=None, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32).reshape(-1, 1, 28, 28)
        #self.X = nn.functional.pad(self.X, (4, 4, 4, 4))
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.augment:
            if random.random() > 0.5:
                x = random_affine(x)
            if random.random() > 0.2:
                x = elastic_distort(x)
            x = x * random.uniform(0.0, 1.1)
            x = x + torch.randn_like(x) * 0.2
            x = x.clamp(0, 1)
        return (x, self.y[idx]) if self.y is not None else x

In [ ]:
import sys
sys.path.append('/kaggle/input/models/elvenmonk/sine-multi-scale-cnn-1390-params-mnist-99/pytorch/default/6')

from SinCNN import SinCNN

model = SinCNN().to(device)

In [ ]:
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total}")

for name, param in model.named_parameters():
    print(f"{name}: shape={list(param.shape)}\n")

import os

path = f'/kaggle/input/models/elvenmonk/sine-multi-scale-cnn-1390-params-mnist-99/pytorch/default/6/SinRCNN_{total}.pth'

if os.path.exists(path):
    model.load_state_dict(torch.load(path, map_location=device))
else:
    print(f'No checkpoint found at {path}')

In [ ]:
train_loader = DataLoader(MNISTDataset(X_train, y_train, augment=True), batch_size=512, shuffle=True, num_workers=0)
valid_loader = DataLoader(MNISTDataset(X_valid, y_valid), batch_size=512, num_workers=2)

optimizer = optim.Adam(model.parameters(), lr=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)
criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    with torch.no_grad():
        correct = sum(
            (model(xb.to(device))[:, :-1].argmax(1) == yb.to(device)).sum().item()
            for xb, yb in valid_loader
        )
    print(f"Epoch {epoch}: {correct}/{len(X_valid)} ({100*correct/len(X_valid):.2f}%)")

In [ ]:
#torch.save(model.state_dict(), path)
test_loader = DataLoader(MNISTDataset(X_test), batch_size=128, num_workers=0)

model.eval()
preds = []
with torch.no_grad():
    for xb in test_loader:
        preds.extend(model(xb.to(device))[:, :-1].argmax(1).cpu().tolist())

pd.DataFrame({'ImageId': range(1, len(preds)+1), 'Label': preds}).to_csv('submission.csv', index=False)

In [ ]:
model.eval()
all_preds, all_labels, all_imgs = [], [], []
with torch.no_grad():
    for xb, yb in valid_loader:
        out = model(xb.to(device))
        all_preds.append(out.argmax(1).cpu())
        all_labels.append(yb)
        all_imgs.append(xb)

preds = torch.cat(all_preds)
labels = torch.cat(all_labels)
imgs = torch.cat(all_imgs)

wrong = (preds != labels).nonzero(as_tuple=True)[0]
print(f"{len(wrong)} wrong out of {len(labels)}")
wrong = wrong[:160]

cols = 16
rows = (len(wrong) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.0, rows * 1.2))
fig.suptitle(f"Misclassified ({len(wrong)})", fontsize=14)
for i, ax in enumerate(axes.flat):
    if i < len(wrong):
        idx = wrong[i]
        ax.imshow(imgs[idx].squeeze(), cmap='gray')
        ax.set_title(f"{preds[idx]}←{labels[idx]}", fontsize=7,
                     color='red')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def show_activations(model, img, true_label):
    model.eval()
    activations = {}

    hooks = []
    for name, module in model.named_modules():
        if name and name not in ('relu', 'pool', 'drop'):
            def make_hook(n):
                def hook(m, inp, out):
                    if n not in activations:
                        activations[n] = []
                    activations[n].append(out.detach().cpu())
                return hook
            hooks.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device)).argmax(1).item()

    for h in hooks:
        h.remove()

    print(f"Predicted: {pred}, True: {true_label}")

    for name, calls in activations.items():
        for call_idx, act in enumerate(calls):
            a = act[0]
            title = f"{name}  {list(a.shape)}" if len(calls) == 1 else f"{name}[{call_idx}]  {list(a.shape)}"
            if a.dim() == 1:
                print(f"{title}  values: {a}")
                continue
            ch = a.shape[0]
            cols = min(ch, 8)
            rows = (ch + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
            fig.suptitle(title)
            axes = np.atleast_2d(axes)
            for i, ax in enumerate(axes.flat):
                if i < ch:
                    ax.imshow(a[i], cmap='viridis')
                ax.axis('off')
            plt.tight_layout()
            plt.show()

# usage: pick a wrong sample by index
show_activations(model, imgs[wrong[0]], labels[wrong[0]].item())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

# label distribution in validation set
axes[0].hist(y_valid, bins=np.arange(11)-0.5, edgecolor='black')
axes[0].set_xticks(range(10))
axes[0].set_title('Validation label distribution')

# digit by index (full dataset)
axes[1].scatter(range(len(y)), y, s=0.1, alpha=0.3)
axes[1].set_title('Label by index (full dataset)')
axes[1].set_ylabel('digit')

plt.tight_layout()
plt.show()

In [ ]:
def plot_matrix(w2d, title):
    fig, ax = plt.subplots(figsize=(max(w2d.shape[1] * 0.4, 2), max(w2d.shape[0] * 0.4, 2)))
    vmax = w2d.abs().max()
    im = ax.imshow(w2d, cmap='RdBu', vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('in'); ax.set_ylabel('out')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

def plot_spatial(w4d, title):
    # w4d: (out_channels, in_channels, H, W)
    out_c, in_c = w4d.shape[:2]
    vmax = w4d.abs().max()

    if in_c == 1:
        # depthwise / single-input — original flat grid
        w = w4d.squeeze(1)
        cols = min(out_c, 8)
        rows = (out_c + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
        fig.suptitle(title)
        axes = np.atleast_2d(axes)
        for i, ax in enumerate(axes.flat):
            if i < out_c:
                ax.imshow(w[i], cmap='RdBu', vmin=-vmax, vmax=vmax)
            ax.axis('off')
    else:
        # multi-channel input — grid of (out × in)
        fig, axes = plt.subplots(out_c, in_c,
                                 figsize=(in_c * 1.2, out_c * 1.2))
        fig.suptitle(f"{title}  (rows=out, cols=in)")
        axes = np.atleast_2d(axes)
        for o in range(out_c):
            for c in range(in_c):
                axes[o, c].imshow(w4d[o, c], cmap='RdBu', vmin=-vmax, vmax=vmax)
                axes[o, c].axis('off')

    plt.tight_layout()
    plt.show()

def visualize_model(model):
    for name, module in model.named_modules():
        if not hasattr(module, 'weight') or module.weight is None:
            continue
        w = module.weight.data.cpu()
        shape = w.shape
        tag = f"{name}  {list(shape)}"

        if isinstance(module, nn.Linear):
            plot_matrix(w, tag)
        elif isinstance(module, (nn.Conv2d, nn.Conv1d)):
            spatial = shape[-1] * shape[-2] if w.dim() == 4 else shape[-1]
            if spatial > 1:
                plot_spatial(w, tag)
            else:
                plot_matrix(w.squeeze(-1).squeeze(-1), tag)

visualize_model(model)